In [1]:
print("Hello")

Hello


Yes. On Windows + VS Code, one of the easiest ways is to use **Scapy** to capture packets and apply a filter for a particular IP such as `8.8.8.8`.

You normally need to run VS Code/terminal **as Administrator**, and Windows needs **Npcap** installed for packet capture.

### 1. Install Scapy

In the VS Code terminal:

```powershell
pip install scapy
```

Also install Npcap if you don't already have Wireshark/Npcap installed.

### 2. Capture packets to/from `8.8.8.8`

Create a file called:

```text
packet_capture.py
```

Put this code in it:

```python
from scapy.all import sniff, IP, TCP, UDP, ICMP
from datetime import datetime

TARGET_IP = "8.8.8.8"


def packet_callback(packet):

    if IP in packet:

        src_ip = packet[IP].src
        dst_ip = packet[IP].dst

        # Only show packets involving TARGET_IP
        if src_ip == TARGET_IP or dst_ip == TARGET_IP:

            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            print("\n-----------------------------------")
            print(f"Time        : {timestamp}")
            print(f"Source IP   : {src_ip}")
            print(f"Destination : {dst_ip}")

            if TCP in packet:
                print("Protocol    : TCP")
                print(f"Source Port : {packet[TCP].sport}")
                print(f"Dest Port   : {packet[TCP].dport}")

            elif UDP in packet:
                print("Protocol    : UDP")
                print(f"Source Port : {packet[UDP].sport}")
                print(f"Dest Port   : {packet[UDP].dport}")

            elif ICMP in packet:
                print("Protocol    : ICMP")

            else:
                print(f"Protocol    : {packet[IP].proto}")


print(f"Starting packet capture for {TARGET_IP}")
print("Press CTRL+C to stop.")

sniff(
    filter=f"host {TARGET_IP}",
    prn=packet_callback,
    store=False
)
```

Run it:

```powershell
python packet_capture.py
```

Then, in another terminal:

```powershell
ping 8.8.8.8
```

You should see something similar to:

```text
Starting packet capture for 8.8.8.8
Press CTRL+C to stop.

-----------------------------------
Time        : 2026-09-02 13:10:15
Source IP   : 192.168.1.25
Destination : 8.8.8.8
Protocol    : ICMP

-----------------------------------
Time        : 2026-09-02 13:10:15
Source IP   : 8.8.8.8
Destination : 192.168.1.25
Protocol    : ICMP
```

So visually:

```text
Your PC
192.168.1.25
    |
    | ICMP Echo Request
    v
 8.8.8.8
Google DNS
    |
    | ICMP Echo Reply
    v
Your PC
192.168.1.25
```

### Capture traffic for a website

For websites, there is an important difference: packet captures operate using **IP addresses**, not website names.

For example:

```text
www.google.com
      ↓
     DNS
      ↓
142.x.x.x
      ↓
TCP/UDP packets
```

Therefore, we can first resolve the website and then capture its IP:

```python
from scapy.all import sniff, IP, TCP, UDP
import socket

WEBSITE = "www.google.com"

target_ip = socket.gethostbyname(WEBSITE)

print(f"Website : {WEBSITE}")
print(f"IP      : {target_ip}")
print("Starting capture...")
print("Press CTRL+C to stop.")


def packet_callback(packet):

    if IP not in packet:
        return

    src = packet[IP].src
    dst = packet[IP].dst

    if src == target_ip or dst == target_ip:

        print("\n==========================")
        print(f"{src} -> {dst}")

        if TCP in packet:
            print(
                f"TCP {packet[TCP].sport} -> "
                f"{packet[TCP].dport}"
            )

        elif UDP in packet:
            print(
                f"UDP {packet[UDP].sport} -> "
                f"{packet[UDP].dport}"
            )


sniff(
    filter=f"host {target_ip}",
    prn=packet_callback,
    store=False
)
```

You could change:

```python
WEBSITE = "www.google.com"
```

to:

```python
WEBSITE = "www.microsoft.com"
```

or:

```python
WEBSITE = "www.cloudflare.com"
```

### Better version: save packets to a `.pcap` file

For network-engineering work, I recommend capturing the packets **and saving them so you can open them in Wireshark**:

```python
from scapy.all import sniff, wrpcap

TARGET_IP = "8.8.8.8"

captured_packets = []


def packet_callback(packet):

    captured_packets.append(packet)

    print(packet.summary())


try:

    print(f"Capturing traffic for {TARGET_IP}")
    print("Press CTRL+C to stop")

    sniff(
        filter=f"host {TARGET_IP}",
        prn=packet_callback,
        store=False
    )

except KeyboardInterrupt:

    print("\nCapture stopped.")

    wrpcap(
        "capture_8.8.8.8.pcap",
        captured_packets
    )

    print(
        f"Saved {len(captured_packets)} packets "
        "to capture_8.8.8.8.pcap"
    )
```

After stopping it with `CTRL+C`, you get:

```text
capture_8.8.8.8.pcap
```

Open that file with Wireshark.

A useful next version for your network-engineering practice would be a single Python packet analyzer where you enter:

```text
Enter IP or website: www.google.com

[1] Capture all traffic
[2] TCP only
[3] UDP only
[4] ICMP only
[5] HTTPS port 443
```

and it automatically shows **source/destination IP, source/destination port, TCP flags, packet size, protocol, timestamps, DNS queries, and saves everything into a PCAP file**.


In [2]:
from scapy.all import sniff, IP, TCP, UDP, ICMP
from datetime import datetime

TARGET_IP = "8.8.8.8"


def packet_callback(packet):

    if IP in packet:

        src_ip = packet[IP].src
        dst_ip = packet[IP].dst

        # Only show packets involving TARGET_IP
        if src_ip == TARGET_IP or dst_ip == TARGET_IP:

            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            print("\n-----------------------------------")
            print(f"Time        : {timestamp}")
            print(f"Source IP   : {src_ip}")
            print(f"Destination : {dst_ip}")

            if TCP in packet:
                print("Protocol    : TCP")
                print(f"Source Port : {packet[TCP].sport}")
                print(f"Dest Port   : {packet[TCP].dport}")

            elif UDP in packet:
                print("Protocol    : UDP")
                print(f"Source Port : {packet[UDP].sport}")
                print(f"Dest Port   : {packet[UDP].dport}")

            elif ICMP in packet:
                print("Protocol    : ICMP")

            else:
                print(f"Protocol    : {packet[IP].proto}")


print(f"Starting packet capture for {TARGET_IP}")
print("Press CTRL+C to stop.")

sniff(
    filter=f"host {TARGET_IP}",
    prn=packet_callback,
    store=False
)

Starting packet capture for 8.8.8.8
Press CTRL+C to stop.

-----------------------------------
Time        : 2026-09-02 14:56:50
Source IP   : 10.0.9.200
Destination : 8.8.8.8
Protocol    : TCP
Source Port : 55201
Dest Port   : 443

-----------------------------------
Time        : 2026-09-02 14:56:50
Source IP   : 8.8.8.8
Destination : 10.0.9.200
Protocol    : TCP
Source Port : 443
Dest Port   : 55201

-----------------------------------
Time        : 2026-09-02 14:56:50
Source IP   : 10.0.9.200
Destination : 8.8.8.8
Protocol    : TCP
Source Port : 55201
Dest Port   : 443

-----------------------------------
Time        : 2026-09-02 14:56:50
Source IP   : 10.0.9.200
Destination : 8.8.8.8
Protocol    : TCP
Source Port : 55201
Dest Port   : 443

-----------------------------------
Time        : 2026-09-02 14:56:50
Source IP   : 8.8.8.8
Destination : 10.0.9.200
Protocol    : TCP
Source Port : 443
Dest Port   : 55201

-----------------------------------
Time        : 2026-09-02 14:56:50
S

<Sniffed: TCP:0 UDP:0 ICMP:0 Other:0>

In [ ]:
from scapy.all import sniff, IP, TCP, UDP
import socket

WEBSITE = "www.google.com"

target_ip = socket.gethostbyname(WEBSITE)

print(f"Website : {WEBSITE}")
print(f"IP      : {target_ip}")
print("Starting capture...")
print("Press CTRL+C to stop.")


def packet_callback(packet):

    if IP not in packet:
        return

    src = packet[IP].src
    dst = packet[IP].dst

    if src == target_ip or dst == target_ip:

        print("\n==========================")
        print(f"{src} -> {dst}")

        if TCP in packet:
            print(
                f"TCP {packet[TCP].sport} -> "
                f"{packet[TCP].dport}"
            )

        elif UDP in packet:
            print(
                f"UDP {packet[UDP].sport} -> "
                f"{packet[UDP].dport}"
            )


sniff(
    filter=f"host {target_ip}",
    prn=packet_callback,
    store=False
)

Website : www.google.com
IP      : 142.251.151.119
Starting capture...
Press CTRL+C to stop.


In [1]:
from scapy.all import sniff, wrpcap

TARGET_IP = "8.8.8.8"

captured_packets = []


def packet_callback(packet):

    captured_packets.append(packet)

    print(packet.summary())


try:

    print(f"Capturing traffic for {TARGET_IP}")
    print("Press CTRL+C to stop")

    sniff(
        filter=f"host {TARGET_IP}",
        prn=packet_callback,
        store=False
    )

except KeyboardInterrupt:

    print("\nCapture stopped.")

    wrpcap(
        "capture_8.8.8.8.pcap",
        captured_packets
    )

    print(
        f"Saved {len(captured_packets)} packets "
        "to capture_8.8.8.8.pcap"
    )

Capturing traffic for 8.8.8.8
Press CTRL+C to stop
Ether / IP / TCP 10.0.9.200:50878 > 8.8.8.8:https S
Ether / IP / TCP 8.8.8.8:https > 10.0.9.200:50878 SA
Ether / IP / TCP 10.0.9.200:50878 > 8.8.8.8:https A
Ether / IP / TCP 10.0.9.200:50878 > 8.8.8.8:https FA
Ether / IP / TCP 8.8.8.8:https > 10.0.9.200:50878 FA / Padding
Ether / IP / TCP 10.0.9.200:50878 > 8.8.8.8:https A
Ether / IP / TCP 10.0.9.200:50879 > 8.8.8.8:https S
Ether / IP / TCP 8.8.8.8:https > 10.0.9.200:50879 SA
Ether / IP / TCP 10.0.9.200:50879 > 8.8.8.8:https A
Ether / IP / TCP 10.0.9.200:50879 > 8.8.8.8:https FA
Ether / IP / TCP 8.8.8.8:https > 10.0.9.200:50879 FA / Padding
Ether / IP / TCP 10.0.9.200:50879 > 8.8.8.8:https A
Ether / IP / TCP 10.0.9.200:50882 > 8.8.8.8:https S
Ether / IP / TCP 8.8.8.8:https > 10.0.9.200:50882 SA
Ether / IP / TCP 10.0.9.200:50882 > 8.8.8.8:https A
Ether / IP / TCP 10.0.9.200:50882 > 8.8.8.8:https FA
Ether / IP / TCP 8.8.8.8:https > 10.0.9.200:50882 FA / Padding
Ether / IP / TCP 10.0.9.20

Yes. Scapy's `sniff()` has a built-in `timeout` parameter. Set `timeout=15`, and the capture will stop after **15 seconds regardless of whether packets are received**.

For your `8.8.8.8` example:

```python
from scapy.all import sniff, IP, TCP, UDP, ICMP
from datetime import datetime

TARGET_IP = "8.8.8.8"
TIMEOUT = 15


def packet_callback(packet):

    if IP in packet:

        src_ip = packet[IP].src
        dst_ip = packet[IP].dst

        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]

        print("\n-----------------------------------")
        print(f"Time        : {timestamp}")
        print(f"Source IP   : {src_ip}")
        print(f"Destination : {dst_ip}")
        print(f"Length      : {len(packet)} bytes")

        if TCP in packet:
            print("Protocol    : TCP")
            print(f"Source Port : {packet[TCP].sport}")
            print(f"Dest Port   : {packet[TCP].dport}")

        elif UDP in packet:
            print("Protocol    : UDP")
            print(f"Source Port : {packet[UDP].sport}")
            print(f"Dest Port   : {packet[UDP].dport}")

        elif ICMP in packet:
            print("Protocol    : ICMP")


print(f"Starting packet capture for {TARGET_IP}")
print(f"Capture will automatically stop after {TIMEOUT} seconds...\n")


packets = sniff(
    filter=f"host {TARGET_IP}",
    prn=packet_callback,
    store=True,
    timeout=TIMEOUT
)


print("\n===================================")
print("CAPTURE FINISHED")
print("===================================")
print(f"Target IP       : {TARGET_IP}")
print(f"Capture duration: {TIMEOUT} seconds")
print(f"Packets captured: {len(packets)}")
```

The important change is simply:

```python
timeout=15
```

So:

```python
sniff(
    filter="host 8.8.8.8",
    prn=packet_callback,
    store=True,
    timeout=15
)
```

### What happens in different situations

If packets are flowing continuously:

```text
0 sec
 |
 | packet
 | packet
 | packet
 | packet
 |
15 sec  ---> STOP
```

It **doesn't reset the timer when a packet arrives**.

If there are no packets at all:

```text
0 sec
 |
 | waiting...
 | waiting...
 | waiting...
 |
15 sec  ---> STOP
```

You'll get:

```text
Starting packet capture for 8.8.8.8
Capture will automatically stop after 15 seconds...

===================================
CAPTURE FINISHED
===================================
Target IP       : 8.8.8.8
Capture duration: 15 seconds
Packets captured: 0
```

If you run:

```powershell
ping 8.8.8.8
```

while the Python script is capturing, you may see:

```text
-----------------------------------
Time        : 2026-09-02 14:59:20.123
Source IP   : 192.168.1.10
Destination : 8.8.8.8
Length      : 74 bytes
Protocol    : ICMP

-----------------------------------
Time        : 2026-09-02 14:59:20.141
Source IP   : 8.8.8.8
Destination : 192.168.1.10
Length      : 74 bytes
Protocol    : ICMP

===================================
CAPTURE FINISHED
===================================
Target IP       : 8.8.8.8
Capture duration: 15 seconds
Packets captured: 8
```

For your network-engineering testing, you can also save whatever was captured during those 15 seconds:

```python
from scapy.all import sniff, wrpcap

TARGET_IP = "8.8.8.8"
TIMEOUT = 15

print(f"Capturing {TARGET_IP} for {TIMEOUT} seconds...")

packets = sniff(
    filter=f"host {TARGET_IP}",
    timeout=TIMEOUT,
    store=True
)

print("Capture finished.")
print(f"Packets captured: {len(packets)}")

if len(packets) > 0:
    wrpcap("capture.pcap", packets)
    print("Packets saved to capture.pcap")
else:
    print("No packets were captured.")
```

Then `capture.pcap` can be opened directly in **Wireshark**.

One distinction worth remembering: `timeout=15` means **total capture duration = 15 seconds**. It does **not** mean "stop if no packets arrive for 15 seconds."







In [2]:
from scapy.all import sniff, IP, TCP, UDP, ICMP
from datetime import datetime

TARGET_IP = "8.8.8.8"
TIMEOUT = 15


def packet_callback(packet):

    if IP in packet:

        src_ip = packet[IP].src
        dst_ip = packet[IP].dst

        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]

        print("\n-----------------------------------")
        print(f"Time        : {timestamp}")
        print(f"Source IP   : {src_ip}")
        print(f"Destination : {dst_ip}")
        print(f"Length      : {len(packet)} bytes")

        if TCP in packet:
            print("Protocol    : TCP")
            print(f"Source Port : {packet[TCP].sport}")
            print(f"Dest Port   : {packet[TCP].dport}")

        elif UDP in packet:
            print("Protocol    : UDP")
            print(f"Source Port : {packet[UDP].sport}")
            print(f"Dest Port   : {packet[UDP].dport}")

        elif ICMP in packet:
            print("Protocol    : ICMP")


print(f"Starting packet capture for {TARGET_IP}")
print(f"Capture will automatically stop after {TIMEOUT} seconds...\n")


packets = sniff(
    filter=f"host {TARGET_IP}",
    prn=packet_callback,
    store=True,
    timeout=TIMEOUT
)


print("\n===================================")
print("CAPTURE FINISHED")
print("===================================")
print(f"Target IP       : {TARGET_IP}")
print(f"Capture duration: {TIMEOUT} seconds")
print(f"Packets captured: {len(packets)}")

Starting packet capture for 8.8.8.8
Capture will automatically stop after 15 seconds...


-----------------------------------
Time        : 2026-09-02 14:59:25.566
Source IP   : 8.8.8.8
Destination : 10.0.9.200
Length      : 66 bytes
Protocol    : TCP
Source Port : 443
Dest Port   : 49626

-----------------------------------
Time        : 2026-09-02 14:59:25.577
Source IP   : 10.0.9.200
Destination : 8.8.8.8
Length      : 54 bytes
Protocol    : TCP
Source Port : 49626
Dest Port   : 443

-----------------------------------
Time        : 2026-09-02 14:59:25.586
Source IP   : 10.0.9.200
Destination : 8.8.8.8
Length      : 54 bytes
Protocol    : TCP
Source Port : 49626
Dest Port   : 443

-----------------------------------
Time        : 2026-09-02 14:59:25.596
Source IP   : 8.8.8.8
Destination : 10.0.9.200
Length      : 60 bytes
Protocol    : TCP
Source Port : 443
Dest Port   : 49626

-----------------------------------
Time        : 2026-09-02 14:59:25.607
Source IP   : 10.0.9.200
Destina

In [3]:
from scapy.all import sniff, wrpcap

TARGET_IP = "8.8.8.8"
TIMEOUT = 15

print(f"Capturing {TARGET_IP} for {TIMEOUT} seconds...")

packets = sniff(
    filter=f"host {TARGET_IP}",
    timeout=TIMEOUT,
    store=True
)

print("Capture finished.")
print(f"Packets captured: {len(packets)}")

if len(packets) > 0:
    wrpcap("capture.pcap", packets)
    print("Packets saved to capture.pcap")
else:
    print("No packets were captured.")

Capturing 8.8.8.8 for 15 seconds...
Capture finished.
Packets captured: 94
Packets saved to capture.pcap


Yes. For a website, the Python script can first **resolve the website hostname to an IP address**, then capture packets involving that IP for exactly **15 seconds**.

For example, you enter `www.google.com`, Python resolves it, captures traffic to/from the resolved IP, and stops automatically after 15 seconds.

```python
from scapy.all import sniff, IP, TCP, UDP, ICMP, wrpcap
from datetime import datetime
import socket

# ==========================================
# SETTINGS
# ==========================================

WEBSITE = "www.google.com"
TIMEOUT = 15


# ==========================================
# RESOLVE WEBSITE TO IP
# ==========================================

try:
    target_ip = socket.gethostbyname(WEBSITE)

    print("======================================")
    print("Website Packet Capture")
    print("======================================")
    print(f"Website       : {WEBSITE}")
    print(f"Resolved IP   : {target_ip}")
    print(f"Capture Time  : {TIMEOUT} seconds")
    print("======================================")

except socket.gaierror:
    print(f"ERROR: Could not resolve {WEBSITE}")
    exit()


# ==========================================
# PACKET CALLBACK
# ==========================================

def packet_callback(packet):

    if IP in packet:

        src_ip = packet[IP].src
        dst_ip = packet[IP].dst

        timestamp = datetime.now().strftime(
            "%Y-%m-%d %H:%M:%S.%f"
        )[:-3]

        print("\n--------------------------------------")
        print(f"Time        : {timestamp}")
        print(f"Source IP   : {src_ip}")
        print(f"Destination : {dst_ip}")
        print(f"Packet Size : {len(packet)} bytes")

        # TCP
        if TCP in packet:

            print("Protocol    : TCP")
            print(f"Source Port : {packet[TCP].sport}")
            print(f"Dest Port   : {packet[TCP].dport}")
            print(f"TCP Flags   : {packet[TCP].flags}")

        # UDP
        elif UDP in packet:

            print("Protocol    : UDP")
            print(f"Source Port : {packet[UDP].sport}")
            print(f"Dest Port   : {packet[UDP].dport}")

        # ICMP
        elif ICMP in packet:

            print("Protocol    : ICMP")

        else:

            print(f"IP Protocol : {packet[IP].proto}")


# ==========================================
# START CAPTURE
# ==========================================

print(f"\nStarting capture for {WEBSITE}")
print(f"Monitoring IP: {target_ip}")
print(f"Capture will stop after {TIMEOUT} seconds.\n")


packets = sniff(
    filter=f"host {target_ip}",
    prn=packet_callback,
    store=True,
    timeout=TIMEOUT
)


# ==========================================
# CAPTURE FINISHED
# ==========================================

print("\n======================================")
print("CAPTURE FINISHED")
print("======================================")

print(f"Website         : {WEBSITE}")
print(f"Resolved IP     : {target_ip}")
print(f"Capture Duration: {TIMEOUT} seconds")
print(f"Packets Captured: {len(packets)}")


# ==========================================
# SAVE PCAP
# ==========================================

if len(packets) > 0:

    filename = "website_capture.pcap"

    wrpcap(filename, packets)

    print(f"PCAP Saved      : {filename}")

else:

    print("No packets were captured.")

print("======================================")
```

### How to test it

Run VS Code/terminal **as Administrator**, then:

```powershell
python website_capture.py
```

You'll initially see:

```text
======================================
Website Packet Capture
======================================
Website       : www.google.com
Resolved IP   : 142.250.191.196
Capture Time  : 15 seconds
======================================

Starting capture for www.google.com
Monitoring IP: 142.250.191.196
Capture will stop after 15 seconds.
```

Now immediately open the website in your browser.

```text
www.google.com
```

The script will capture for:

```text
          15-second capture window

0 sec ─────────────────────────────── 15 sec
  │                                      │
  │ Browser → Google                     │
  │        TCP/UDP packets               │
  │        HTTPS traffic                 │
  │ Google → Browser                     │
  │                                      │
  └──────────────────────────────────────┘
                                         ↓
                                      STOP
```

Even if packets are continuously flowing, it stops at 15 seconds.

### Make it accept any website

Instead of hardcoding:

```python
WEBSITE = "www.google.com"
```

you can make the script ask you:

```python
WEBSITE = input("Enter website: ")
TIMEOUT = 15
```

Then:

```text
Enter website: www.microsoft.com
```

or:

```text
Enter website: www.google.com
```

or:

```text
Enter website: www.cloudflare.com
```

One caveat: modern websites often use **multiple IP addresses/CDNs**, and browsers may use IPv6 or QUIC/HTTP/3. `socket.gethostbyname()` captures only one resolved IPv4 address, so it may not capture **all** traffic associated with a website. If your goal is Wireshark-like website analysis, the next improvement would be to resolve **all IPv4 + IPv6 addresses** for the hostname and capture all of them during the 15-second window.


In [4]:
from scapy.all import sniff, IP, TCP, UDP, ICMP, wrpcap
from datetime import datetime
import socket

# ==========================================
# SETTINGS
# ==========================================

WEBSITE = "www.google.com"
TIMEOUT = 15


# ==========================================
# RESOLVE WEBSITE TO IP
# ==========================================

try:
    target_ip = socket.gethostbyname(WEBSITE)

    print("======================================")
    print("Website Packet Capture")
    print("======================================")
    print(f"Website       : {WEBSITE}")
    print(f"Resolved IP   : {target_ip}")
    print(f"Capture Time  : {TIMEOUT} seconds")
    print("======================================")

except socket.gaierror:
    print(f"ERROR: Could not resolve {WEBSITE}")
    exit()


# ==========================================
# PACKET CALLBACK
# ==========================================

def packet_callback(packet):

    if IP in packet:

        src_ip = packet[IP].src
        dst_ip = packet[IP].dst

        timestamp = datetime.now().strftime(
            "%Y-%m-%d %H:%M:%S.%f"
        )[:-3]

        print("\n--------------------------------------")
        print(f"Time        : {timestamp}")
        print(f"Source IP   : {src_ip}")
        print(f"Destination : {dst_ip}")
        print(f"Packet Size : {len(packet)} bytes")

        # TCP
        if TCP in packet:

            print("Protocol    : TCP")
            print(f"Source Port : {packet[TCP].sport}")
            print(f"Dest Port   : {packet[TCP].dport}")
            print(f"TCP Flags   : {packet[TCP].flags}")

        # UDP
        elif UDP in packet:

            print("Protocol    : UDP")
            print(f"Source Port : {packet[UDP].sport}")
            print(f"Dest Port   : {packet[UDP].dport}")

        # ICMP
        elif ICMP in packet:

            print("Protocol    : ICMP")

        else:

            print(f"IP Protocol : {packet[IP].proto}")


# ==========================================
# START CAPTURE
# ==========================================

print(f"\nStarting capture for {WEBSITE}")
print(f"Monitoring IP: {target_ip}")
print(f"Capture will stop after {TIMEOUT} seconds.\n")


packets = sniff(
    filter=f"host {target_ip}",
    prn=packet_callback,
    store=True,
    timeout=TIMEOUT
)


# ==========================================
# CAPTURE FINISHED
# ==========================================

print("\n======================================")
print("CAPTURE FINISHED")
print("======================================")

print(f"Website         : {WEBSITE}")
print(f"Resolved IP     : {target_ip}")
print(f"Capture Duration: {TIMEOUT} seconds")
print(f"Packets Captured: {len(packets)}")


# ==========================================
# SAVE PCAP
# ==========================================

if len(packets) > 0:

    filename = "website_capture.pcap"

    wrpcap(filename, packets)

    print(f"PCAP Saved      : {filename}")

else:

    print("No packets were captured.")

print("======================================")

Website Packet Capture
Website       : www.google.com
Resolved IP   : 142.251.152.119
Capture Time  : 15 seconds

Starting capture for www.google.com
Monitoring IP: 142.251.152.119
Capture will stop after 15 seconds.


CAPTURE FINISHED
Website         : www.google.com
Resolved IP     : 142.251.152.119
Capture Duration: 15 seconds
Packets Captured: 0
No packets were captured.


Yes — here is the improved version that resolves **all IPv4 and IPv6 addresses** for a website, captures traffic to any of them, and stops automatically after **15 seconds**.

```python
from scapy.all import sniff, IP, IPv6, TCP, UDP, wrpcap
from datetime import datetime
import socket

WEBSITE = input("Enter website: ").strip()
TIMEOUT = 15


def resolve_all_addresses(hostname):
    ipv4 = set()
    ipv6 = set()

    results = socket.getaddrinfo(hostname, None)

    for result in results:
        family = result[0]
        address = result[4][0]

        if family == socket.AF_INET:
            ipv4.add(address)

        elif family == socket.AF_INET6:
            ipv6.add(address)

    return sorted(ipv4), sorted(ipv6)


try:
    ipv4_addresses, ipv6_addresses = resolve_all_addresses(WEBSITE)

except socket.gaierror:
    print(f"Could not resolve website: {WEBSITE}")
    raise SystemExit


print("\n======================================")
print("Website Packet Capture")
print("======================================")
print(f"Website      : {WEBSITE}")
print(f"Capture time : {TIMEOUT} seconds")

print("\nIPv4 addresses:")
for ip in ipv4_addresses:
    print(f"  {ip}")

print("\nIPv6 addresses:")
for ip in ipv6_addresses:
    print(f"  {ip}")

print("======================================\n")


all_target_ips = set(ipv4_addresses + ipv6_addresses)


def packet_callback(packet):

    src = None
    dst = None

    if IP in packet:
        src = packet[IP].src
        dst = packet[IP].dst

    elif IPv6 in packet:
        src = packet[IPv6].src
        dst = packet[IPv6].dst

    else:
        return

    if src not in all_target_ips and dst not in all_target_ips:
        return

    timestamp = datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S.%f"
    )[:-3]

    print("\n--------------------------------------")
    print(f"Time        : {timestamp}")
    print(f"Source IP   : {src}")
    print(f"Destination : {dst}")
    print(f"Packet Size : {len(packet)} bytes")

    if TCP in packet:

        print("Protocol    : TCP")
        print(f"Source Port : {packet[TCP].sport}")
        print(f"Dest Port   : {packet[TCP].dport}")
        print(f"TCP Flags   : {packet[TCP].flags}")

    elif UDP in packet:

        print("Protocol    : UDP")
        print(f"Source Port : {packet[UDP].sport}")
        print(f"Dest Port   : {packet[UDP].dport}")

        if packet[UDP].sport == 443 or packet[UDP].dport == 443:
            print("Possible    : QUIC / HTTP3")

    else:

        print("Protocol    : Other")


# Build BPF filter
filters = []

for ip in ipv4_addresses:
    filters.append(f"host {ip}")

for ip in ipv6_addresses:
    filters.append(f"host {ip}")

bpf_filter = " or ".join(filters)


if not bpf_filter:
    print("No IP addresses found.")
    raise SystemExit


print(f"Starting capture for {WEBSITE}")
print(f"Capture will stop automatically after {TIMEOUT} seconds.")
print("Open or refresh the website now.\n")


packets = sniff(
    filter=bpf_filter,
    prn=packet_callback,
    store=True,
    timeout=TIMEOUT
)


print("\n======================================")
print("CAPTURE FINISHED")
print("======================================")

print(f"Website          : {WEBSITE}")
print(f"Capture duration : {TIMEOUT} seconds")
print(f"Packets captured : {len(packets)}")


if len(packets) > 0:

    filename = "website_capture.pcap"

    wrpcap(filename, packets)

    print(f"PCAP saved       : {filename}")

else:

    print("No packets captured.")

print("======================================")
```

Run it in VS Code:

```powershell
python website_capture.py
```

Then enter something like:

```text
Enter website: www.google.com
```

You may see:

```text
Website      : www.google.com
Capture time : 15 seconds

IPv4 addresses:
  142.250.191.100
  142.250.191.101
  142.250.191.102

IPv6 addresses:
  2607:f8b0:4005:805::2004
```

Then open or refresh `www.google.com` in your browser during that 15-second period.

This version can capture traffic like:

```text
Your PC
   |
   | TCP 443
   | UDP 443 / QUIC
   v
Google IPv4/IPv6
```

and after exactly:

```text
0 sec -------------------------- 15 sec
                                  |
                                  STOP
```

it saves:

```text
website_capture.pcap
```

which you can open directly in Wireshark.

One important point: this captures traffic to the IP addresses returned for the hostname you enter. A webpage can also contact many additional domains such as CDNs, analytics, images, APIs, ads, authentication services, etc., so this still may not represent **every packet generated while loading the webpage**.

If you want, I can next give you a version that captures **all packets generated by your PC during the 15 seconds and automatically identifies which domains/IPs the browser contacted**, which is closer to a mini-Wireshark tool.


In [5]:
from scapy.all import sniff, IP, IPv6, TCP, UDP, wrpcap
from datetime import datetime
import socket

WEBSITE = input("Enter website: ").strip()
TIMEOUT = 15


def resolve_all_addresses(hostname):
    ipv4 = set()
    ipv6 = set()

    results = socket.getaddrinfo(hostname, None)

    for result in results:
        family = result[0]
        address = result[4][0]

        if family == socket.AF_INET:
            ipv4.add(address)

        elif family == socket.AF_INET6:
            ipv6.add(address)

    return sorted(ipv4), sorted(ipv6)


try:
    ipv4_addresses, ipv6_addresses = resolve_all_addresses(WEBSITE)

except socket.gaierror:
    print(f"Could not resolve website: {WEBSITE}")
    raise SystemExit


print("\n======================================")
print("Website Packet Capture")
print("======================================")
print(f"Website      : {WEBSITE}")
print(f"Capture time : {TIMEOUT} seconds")

print("\nIPv4 addresses:")
for ip in ipv4_addresses:
    print(f"  {ip}")

print("\nIPv6 addresses:")
for ip in ipv6_addresses:
    print(f"  {ip}")

print("======================================\n")


all_target_ips = set(ipv4_addresses + ipv6_addresses)


def packet_callback(packet):

    src = None
    dst = None

    if IP in packet:
        src = packet[IP].src
        dst = packet[IP].dst

    elif IPv6 in packet:
        src = packet[IPv6].src
        dst = packet[IPv6].dst

    else:
        return

    if src not in all_target_ips and dst not in all_target_ips:
        return

    timestamp = datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S.%f"
    )[:-3]

    print("\n--------------------------------------")
    print(f"Time        : {timestamp}")
    print(f"Source IP   : {src}")
    print(f"Destination : {dst}")
    print(f"Packet Size : {len(packet)} bytes")

    if TCP in packet:

        print("Protocol    : TCP")
        print(f"Source Port : {packet[TCP].sport}")
        print(f"Dest Port   : {packet[TCP].dport}")
        print(f"TCP Flags   : {packet[TCP].flags}")

    elif UDP in packet:

        print("Protocol    : UDP")
        print(f"Source Port : {packet[UDP].sport}")
        print(f"Dest Port   : {packet[UDP].dport}")

        if packet[UDP].sport == 443 or packet[UDP].dport == 443:
            print("Possible    : QUIC / HTTP3")

    else:

        print("Protocol    : Other")


# Build BPF filter
filters = []

for ip in ipv4_addresses:
    filters.append(f"host {ip}")

for ip in ipv6_addresses:
    filters.append(f"host {ip}")

bpf_filter = " or ".join(filters)


if not bpf_filter:
    print("No IP addresses found.")
    raise SystemExit


print(f"Starting capture for {WEBSITE}")
print(f"Capture will stop automatically after {TIMEOUT} seconds.")
print("Open or refresh the website now.\n")


packets = sniff(
    filter=bpf_filter,
    prn=packet_callback,
    store=True,
    timeout=TIMEOUT
)


print("\n======================================")
print("CAPTURE FINISHED")
print("======================================")

print(f"Website          : {WEBSITE}")
print(f"Capture duration : {TIMEOUT} seconds")
print(f"Packets captured : {len(packets)}")


if len(packets) > 0:

    filename = "website_capture.pcap"

    wrpcap(filename, packets)

    print(f"PCAP saved       : {filename}")

else:

    print("No packets captured.")

print("======================================")


Website Packet Capture
Website      : yahoo.com
Capture time : 15 seconds

IPv4 addresses:
  74.6.143.25
  74.6.143.26
  74.6.231.20
  74.6.231.21
  98.137.11.163
  98.137.11.164

IPv6 addresses:

Starting capture for yahoo.com
Capture will stop automatically after 15 seconds.
Open or refresh the website now.


CAPTURE FINISHED
Website          : yahoo.com
Capture duration : 15 seconds
Packets captured : 0
No packets captured.
